# Mesh cutting (Japan / Kanto)
This notebook applies the fault cutting hierarchy to pre-generated triangular mesh surfaces, trimming lower-priority faults where they intersect higher-priority faults. A depth raster is also used to trim meshes at depth.

## Imports

In [1]:
from fault_mesh.faults.leapfrog import LeapfrogMultiFault
from fault_mesh.faults.mesh import FaultMesh
from fault_mesh.io.array_operations import read_raster
import pyvista as pv
import os

## Read fault data
Set the output coordinate system EPSG code and read fault data from the GeoJSON file, then find connections between segments.

In [2]:
# Set output coordinate system EPSG code (set to None to disable reprojection)
epsg = 32654

# Read fault data from GeoJSON file
fault_fname = r"fault_Kanto\fault_Kanto\Kanto_251216_Emori_crdchanged_FaultstatAdded_densified_500.geojson"

# Build fault network: set segment connection tolerance and find connections between segments
dist_tolerance = 1000.
fault_data = LeapfrogMultiFault.from_nz_cfm_shp(fault_fname, remove_colons=True, epsg=epsg, smoothing_n=None, dip_multiplier=1.0, exclude_zero=False)
fault_data.segment_distance_tolerance = dist_tolerance
fault_data.find_connections(verbose=False)

missing expected field
missing expected field
missing expected field


Found 5 connections
Found 4 connections between segment ends


## Define fault systems and cutting hierarchy
Read the fault system definitions and generate curated multi-segment faults, then read the cutting hierarchy.

In [3]:
# Read fault system definitions and generate curated (multi-segment) faults
fault_data.read_fault_systems(r"fault_Kanto\fault_Kanto\Kanto_251216_Emori_251226_connected_v1_suggested.csv")
fault_data.generate_curated_faults()

# Read fault cutting hierarchy to determine termination relationships
fault_data.read_cutting_hierarchy(r"fault_Kanto\fault_Kanto\Kanto_251216_Emori_251226_hierarchy_v1_suggested.csv")

Creating connected fault system: Fukaya - Takasaki combined with segments: ['Fukaya', 'Takasaki']
Setting Takasaki as first segment
Creating connected fault system: Susugatani - Isehara combined with segments: ['Isehara', 'Susugatani']
Setting Susugatani as first segment
Creating connected fault system: Tachikawa-B - Tachikawa-A combined with segments: ['Tachikawa-A', 'Tachikawa-B']
Setting Tachikawa-B as first segment


## Read depth raster
Load the depth raster (UTM-projected) which will be used to trim meshes at depth.

In [4]:
# Read depth raster and convert to PyVista surface
depth_pyvista = read_raster("japan_smoothed_metres_UTM.tif", use_z=True, out_crs=f"EPSG:{epsg}")

## Load pre-generated meshes
Read existing OBJ mesh files (generated by `uncut_meshes_for_all_surfaces.ipynb`) and attach them to the curated fault objects.

In [5]:
# Read in existing meshes for each fault if they exist, and store in a dictionary for cutting
for fault in fault_data.curated_faults:
    obj_file = os.path.join(r"test_objs", f"{fault.name}_depth_contours.obj")
    if os.path.exists(obj_file):
        print(f"Reading existing mesh for {fault.name}")
        mesh = FaultMesh.from_file(obj_file)
        fault.mesh = mesh

# Dictionary for use with cutting hierarchy
cutting_dict = {fault.name: fault for fault in fault_data.curated_faults if fault.mesh is not None}

Reading existing mesh for Fukaya - Takasaki combined
Reading existing mesh for Susugatani - Isehara combined
Reading existing mesh for Tachikawa-B - Tachikawa-A combined
Reading existing mesh for Ayasegawa
Reading existing mesh for Hirai
Reading existing mesh for Kamogawachikotai-minami
Reading existing mesh for Kannawa
Reading existing mesh for Kitatake
Reading existing mesh for Kouzu-Matsuda
Reading existing mesh for Kurokura
Reading existing mesh for Minamishitaura
Reading existing mesh for Sekiya
Reading existing mesh for Shibusawa


## Cut meshes according to hierarchy
For each fault, cut its mesh against all higher-priority faults where an intersection is detected, then trim at depth using the depth raster. Output cut meshes are written to the `test_final_meshes` directory.

In [6]:
# Create output directory for final cut meshes
output_dir = r"test_final_meshes"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Threshold currently does nothing, ignore for now.
threshold = 0.7
# Set a minimum distance threshold to avoid cutting faults that are very close together
# (e.g. due to KDTree artefacts or small faults right next to each other)
min_distance = 2.e3

for fault_name in fault_data.cutting_hierarchy:
    print(f"Processing {fault_name}...")
    fault = fault_data.name_dict[fault_name]
    # Get list of faults that cut this fault, in order of hierarchy
    faults_to_cut = fault_data.cutting_hierarchy[:fault_data.cutting_hierarchy.index(fault_name)]
    # Loop through cutting faults and decide whether to cut based on intersection
    # with cutting fault mesh and distance to cutting fault trace
    for cut_name in faults_to_cut:
        if cut_name in cutting_dict:
            higher_faults_to_cut = fault_data.cutting_hierarchy[:fault_data.cutting_hierarchy.index(cut_name)]
            higher_meshes = [cutting_dict[name].mesh for name in higher_faults_to_cut]
            cutting_mesh = cutting_dict[cut_name].mesh
            decide_to_cut = fault.mesh.decide_whether_to_cut(cutting_mesh, threshold=threshold, min_distance=min_distance, higher_meshes=higher_meshes, bottom_depth=-31500.)
            if decide_to_cut:
                print(f"Cutting {fault.name} by {cut_name}")
                cutting_fragment = cutting_mesh.generate_cutting_mesh(fault.mesh, max_distance=5.e3)
                new_mesh = fault.mesh.cut_mesh(cutting_mesh, fault_trace=fault.original_nztm_trace_array, cutting_fragment=cutting_fragment)
                fault_data.name_dict[fault.name].mesh = new_mesh
                cutting_dict[fault.name].mesh = new_mesh

    # Cut meshes by depth surface and save final meshes as OBJ files
    try:
        depth_trimmed = fault_data.name_dict[fault.name].mesh.cut_mesh_pv(depth_pyvista, fault_trace=fault.original_nztm_trace_array)
        fault_data.name_dict[fault.name].mesh = depth_trimmed
    except Exception as e:
        print(f"Error trimming depth for {fault.name}: {e}")
        continue
    output_file = os.path.join(output_dir, f"{fault.name}_cut.obj")
    fault_data.name_dict[fault.name].mesh.mesh.write(output_file)
    print(f"Done: {fault.name}")

Processing Kouzu-Matsuda...
Clipped2 has no surface points for Kouzu-Matsuda_depth_contours. Returning clipped1.
Done: Kouzu-Matsuda
Processing Sekiya...
Clipped2 has no surface points for Sekiya_depth_contours. Returning clipped1.
Done: Sekiya
Processing Kitatake...
Clipped2 has no surface points for Kitatake_depth_contours. Returning clipped1.
Done: Kitatake
Processing Kannawa...
Clipped2 has no surface points for Kannawa_depth_contours. Returning clipped1.
Done: Kannawa
Processing Shibusawa...
Cutting Shibusawa by Kouzu-Matsuda
Clipped2 resulted in no surface points for Shibusawa_depth_contours after cutting with Kouzu-Matsuda_depth_contours. Returning clipped1.
Clipped2 has no surface points for Shibusawa_depth_contours. Returning clipped1.
Done: Shibusawa
Processing Minamishitaura...
Clipped2 has no surface points for Minamishitaura_depth_contours. Returning clipped1.
Done: Minamishitaura
Processing Susugatani - Isehara combined...
Clipped2 has no surface points for Susugatani - I